# 📝 Lecture 7 Activity Notebook: Text as Data
## AA640: Data Analytics and Text Mining
### Bryant University | Prof. Gianluca Brero

---

**IMPORTANT:** *Before starting, save a copy to your Drive via `File > "Save a copy in Drive"`*

### 🎯 Estimated Time: 90 Minutes

#### In-Class Schedule

| Time | Clock | Block |
|------|-------|-------|
| 30 min | 6:30pm | Lecture: Text as Data & Regex |
| 20 min | 7:00pm | **Activity 1** — Regex patterns on real text |
| 30 min | 7:20pm | Lecture: Tokenization & Preprocessing |
| 30 min | 7:50pm | Break + Quiz |
| 20 min | 8:20pm | **Activity 2** — Build a text-cleaning pipeline |
| 30 min | 8:40pm | Open time — questions, finish notebook |

#### After-Class Activities (~30 min)
*Complete these on your own after lecture.*

| Activity | Topic | Time |
|----------|-------|------|
| Activity 3 | Regex Challenges | ~10 min |
| Activity 4 | Web Scraping + Text Pipeline | ~20 min |

### ⚙️ How to Use This Notebook
1. **Read** each section carefully
2. **Run** the example cells first to see how things work
3. **Complete** the activities marked with 📝
4. **Check** your answers against the expected output

> 💡 **Tip:** If you get stuck, re-read the example cell right above the activity — the pattern is always there!

## Setup: Install & Import Libraries

Run these two cells first. They install and import everything we need for today.

- **`pandas`** — for working with DataFrames (you already know this from L5)
- **`re`** — Python's built-in module for regular expressions (pattern matching)
- **`Counter`** — a handy tool for counting how often things appear in a list
- **`nltk`** — Natural Language Toolkit, a library for working with text

In [ ]:
# This installs the NLTK library if it's not already installed.
# The exclamation mark (!) means "run this as a terminal command, not Python code."
!pip install nltk

In [ ]:
import pandas as pd
import re
from collections import Counter

import nltk
nltk.download('punkt_tab', quiet=True)   # needed for word_tokenize
nltk.download('stopwords', quiet=True)    # needed for stopword lists

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Load the list of English stopwords (common words like "the", "is", "and")
# We use set() because checking membership in a set is much faster than a list
stop_words = set(stopwords.words('english'))

# Create a stemmer -- it reduces words to their root form
# e.g., "running" -> "run", "products" -> "product"
stemmer = PorterStemmer()

print("Setup complete!")
print(f"Number of English stopwords: {len(stop_words)}")
print(f"Some examples: {sorted(list(stop_words))[:15]}")
print(f"\nStemmer demo: 'running' -> '{stemmer.stem('running')}', 'products' -> '{stemmer.stem('products')}'")

---
# 📌 IN-CLASS ACTIVITIES

Complete these sections during class time. Each activity has demo cells above it — run the demos first, then work on the 📝 activity.

---

## Activity 1: Regex Patterns on Real Text (~20 min)

*Your manager drops a spreadsheet of customer reviews on your desk: "There are phone numbers and emails buried in these reviews — extract them all so we can follow up with customers. Also strip out all the punctuation, and pull out every 4-digit year people mention."*

In L5 we used `.replace('$', '')` to remove exact characters we knew about. But what if you don't know exactly what's in the text? **Regular expressions** (regex) let you describe a *pattern* instead of an exact string.

### Demo: Regex with `re.findall()` and `re.sub()`

**Quick recap from the lecture slides:**

| Pattern | What it matches | Example |
|---------|---------|--------|
| `\d` | Any single digit (0–9) | `\d\d\d` matches `204` |
| `\w` | Any letter, digit, or `_` | `\w\w\w` matches `cat` |
| `[abc]` | Any one of these characters | `[aeiou]` matches any vowel |
| `[^abc]` | Any character **NOT** in the set | `[^0-9]` matches non-digits |
| `+` | One or more of the previous | `\d+` matches `1`, `42`, `2026` |
| `{n}` | Exactly n of the previous | `\d{4}` matches a 4-digit year |

**Two key functions:**
- **`re.findall(pattern, text)`** — finds all matches and returns them as a list
- **`re.sub(pattern, replacement, text)`** — replaces all matches with something else

**Important:** Always put `r` before your pattern string (e.g., `r'\d+'`). This tells Python to treat backslashes literally instead of as special characters.

In [ ]:
# Demo: Extract patterns with re.findall()
#
# re.findall() scans through the text and returns every piece that matches
# your pattern. Think of it like a super-powered Ctrl+F that can search
# for patterns, not just exact words.

text = "Founded in 1985, the company expanded in 2010 and went public in 2023."

# \d{4} means "exactly 4 digits in a row" -- this finds years!
years = re.findall(r'\d{4}', text)
print("Years found:", years)

# \w+ means "one or more word characters" -- this finds every word
words = re.findall(r'\w+', text)
print("Words found:", words)

In [ ]:
# Demo: Remove punctuation with re.sub()
#
# re.sub() finds everything matching the pattern and replaces it.
# Here we replace with '' (empty string), which effectively deletes the match.
#
# The pattern [^a-zA-Z\s] means:
#   [ ]      = "any character in this set"
#   ^        = "NOT" (when inside brackets)
#   a-zA-Z   = letters (lowercase and uppercase)
#   \s       = whitespace (spaces, tabs)
#
# So [^a-zA-Z\s] = "anything that is NOT a letter or space" = punctuation!

review = "Wow!!! This product is AMAZING... 5/5 stars!!!"
clean = re.sub(r'[^a-zA-Z\s]', '', review)
print("Original:", review)
print("Cleaned: ", clean)

In [ ]:
# Demo: Extract email addresses
#
# Email pattern breakdown:
#   \w+    = one or more word characters (the username part)
#   @      = the literal @ symbol
#   \w+    = one or more word characters (the domain name)
#   \.     = a literal dot (we need \ because . alone means "any character")
#   \w+    = one or more word characters (com, edu, org, etc.)

text = "Contact alice@corp.com or bob@school.edu for details"
emails = re.findall(r'\w+@\w+\.\w+', text)
print("Emails found:", emails)

In [ ]:
# Demo: Using regex with pandas
#
# Just like .str.lower() or .str.strip() from L5, you can use regex
# with the .str accessor to apply patterns to an entire column at once.
#
# Key methods:
#   .str.replace(pattern, replacement, regex=True)  -- find & replace
#   .str.findall(pattern)                           -- extract all matches
#   .str.contains(pattern)                          -- True/False for each row

df_demo = pd.DataFrame({
    'review': [
        'Great product! Bought in 2022.',
        'Terrible... broke after 2 weeks.',
        'Love it!!! Best purchase of 2023.'
    ]
})

# Remove punctuation from the whole column
# regex=True tells pandas to treat the first argument as a regex pattern
df_demo['clean'] = df_demo['review'].str.replace(r'[^\w\s]', '', regex=True)

# Extract years from each review (returns a list per row)
df_demo['years'] = df_demo['review'].str.findall(r'\d{4}')

print(df_demo[['review', 'clean', 'years']])

### 📝 Activity 1a: Extract Years from Reviews

Use regex to extract all 4-digit years from the `description` column.

**What to do:** Create a new column called `'years'` that contains a list of all years found in each row.

*Hint: Look at the last demo cell above — we used `.str.findall(r'\d{4}')` to do exactly this.*

In [ ]:
import pandas as pd
events = pd.DataFrame({
    'description': [
        'Company was founded in 1998 and went public in 2005.',
        'Merged with TechCorp in 2018.',
        'Opened 3 offices between 2020 and 2023.',
        'No year mentioned here.',
        'Revenue doubled from 2015 to 2019, then tripled by 2024.'
    ]
})

print("Descriptions:")
for desc in events['description']:
    print(f"  {desc}")
print()

# TODO: Create a new column 'years' containing a list of all 4-digit years found
# events['years'] = events['description'].str.findall(r'\d{4}')

events['years'] = events['description'].str.findall(r'\d{4}')

# Expected output:
# 0        [1998, 2005]
# 1              [2018]
# 2        [2020, 2023]
# 3                  []
# 4    [2015, 2019, 2024]

print("Extracted years:")
print(events['years'])

### 📝 Activity 1b: Remove Punctuation from Reviews

Use regex with pandas to remove all punctuation from the `review` column. Store the result in a new column called `'clean'`.

**What to do:** Use `.str.replace()` with the pattern `r'[^\w\s]'` (anything that is NOT a letter, digit, or space) and replace it with `''` (nothing).

*Hint: `.str.replace(r'[^\w\s]', '', regex=True)`*

In [ ]:
reviews_df = pd.DataFrame({
    'review': [
        'Great product!!! Loved it.',
        'Terrible... would NOT buy again!!!',
        'Okay-ish. 3/5 stars.',
        'WOW! Best purchase EVER (seriously).',
        "Can't believe how bad this is... $50 wasted!"
    ]
})

print("Before:")
print(reviews_df['review'].tolist())
print()

# TODO: Create a 'clean' column with all punctuation removed
# reviews_df['clean'] = reviews_df['review'].str.replace(r'[^\w\s]', '', regex=True)

reviews_df['clean'] = reviews_df['review'].str.replace(r'[^\w\s]', '', regex=True)

# Expected output:
# 'Great product Loved it'
# 'Terrible would NOT buy again'
# 'Okayish 35 stars'
# 'WOW Best purchase EVER seriously'
# 'Cant believe how bad this is 50 wasted'

print("After:")
print(reviews_df['clean'].tolist())

### 📝 Activity 1c: Extract Emails from Text

Use `re.findall()` to extract all email addresses from the text below.

**What to do:** Use the email pattern `r'\w+@\w+\.\w+'` which matches:
- `\w+` — one or more word characters (the username)
- `@` — the @ symbol
- `\w+` — one or more word characters (the domain)
- `\.` — a literal dot
- `\w+` — the extension (com, edu, org, etc.)

In [ ]:
message = """Please contact support@company.com for billing issues.
For technical help, email tech@helpdesk.org instead.
General inquiries can go to info@company.com.
Our CEO (jane@executive.net) is also available."""

# TODO: Extract all email addresses
# emails = re.findall(r'\w+@\w+\.\w+', message)

emails = re.findall(r'\w+@\w+\.\w+', message)

print(f"Found {len(emails)} emails")
for e in emails:
  print(f"  {e}")

# Expected: ['support@company.com', 'tech@helpdesk.org',
#            'info@company.com', 'jane@executive.net']

# print(f"Found {len(emails)} emails:")
# for e in emails:
#     print(f"  {e}")

---
## Activity 2: Build a Text-Cleaning Pipeline (~20 min)

*Your manager comes back and says: "Can you build a text-cleaning function? I need lowercase, no punctuation, and no filler words like 'the' and 'is'. Test it on some reviews and show me the before and after. Compare the words with and without removing stopwords — show me both so I can see the difference."*

In this activity we'll build a complete text-processing pipeline, step by step.

### Demo: Tokenization and Stopword Removal

**Tokenization** = splitting text into individual words (called "tokens").

We use NLTK's `word_tokenize()` instead of Python's `.split()` because it's smarter about punctuation:
- `"don't".split()` → `["don't"]` (one token — not helpful)
- `word_tokenize("don't")` → `["do", "n't"]` (two tokens — separates the negation)

**Stopwords** = extremely common words like "the", "is", "and" that appear in every text. If we count words without removing them, the top words will always be these filler words instead of the interesting ones.

In [ ]:
# Demo: Why stopword removal matters
#
# Let's tokenize a sentence and see what the most common words are.

text = "The product is absolutely great and I would definitely recommend it to everyone"

# Step 1: Tokenize (split into individual words)
# We also lowercase first so "The" and "the" count as the same word
tokens = word_tokenize(text.lower())
print("All tokens:", tokens)
print(f"Count: {len(tokens)} words")
print()

# Step 2: Remove stopwords
# This list comprehension keeps only words that are NOT in our stopwords set
# It reads: "keep w for each w in tokens, but only if w is not a stopword"
filtered = [w for w in tokens if w not in stop_words]
print("After removing stopwords:", filtered)
print(f"Count: {len(filtered)} words")
print()
print("Notice: 'the', 'is', 'and', 'I', 'would', 'it', 'to' are gone.")
print("What's left are the meaningful words: 'absolutely', 'great', 'definitely', 'recommend', 'everyone'")

In [ ]:
# Demo: Stemming & Lemmatization
#
# After removing stopwords, we still have a problem:
# "running", "runs", and "ran" are the same idea, but Python counts them
# as three different words. We need to reduce them to a common root.
#
# Two approaches:
#   1. Stemming  -- strips suffixes (-ing, -ed, -s) using simple rules
#                   Fast, but sometimes produces non-words
#   2. Lemmatization -- looks up the actual dictionary form
#                       More accurate, but slower and needs more setup

from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

words = ['running', 'runs', 'studies', 'products']

print("STEMMING (rule-based -- chop suffixes):")
for w in words:
    print(f"  {w:12s} -> {stemmer.stem(w)}")

print()

# Lemmatization example (for comparison -- we won't use this in our pipeline)
# You have to tell it whether the word is a verb ('v'), noun ('n'), etc.
# otherwise it can't look up the right form.
from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('wordnet', quiet=True)
lemmatizer = WordNetLemmatizer()

print("LEMMATIZATION (dictionary lookup -- always real words):")
for w in words:
    print(f"  {w:12s} -> {lemmatizer.lemmatize(w, pos='v')}")

print()
print("Lemmatization gives cleaner results (e.g., 'study' not 'studi'),")
print("but you need to specify the part of speech (verb, noun, etc.).")
print("For this course we'll use STEMMING -- simpler and good enough for word counting.")

In [ ]:
# Demo: A reusable clean_text() function
#
# Instead of doing each step manually every time, we combine them
# into one function we can reuse. This function:
#   1. Lowercases everything (so "GREAT" and "great" are treated the same)
#   2. Removes punctuation (so "great!" becomes "great")
#   3. Splits into individual words (tokenization)
#   4. Removes common filler words (stopwords)
#   5. Stems each word to its root (so "products" and "production" match)

def clean_text(text):
    text = text.lower()                          # step 1: lowercase
    text = re.sub(r'[^\w\s]', '', text)          # step 2: remove punctuation
    tokens = word_tokenize(text)                  # step 3: split into words
    tokens = [w for w in tokens if w not in stop_words]  # step 4: remove stopwords
    tokens = [stemmer.stem(w) for w in tokens]    # step 5: stem to root form
    return tokens

# Test it out:
result = clean_text("The products are GREAT!!! Highly recommended.")
print("Input:  'The products are GREAT!!! Highly recommended.'")
print("Output:", result)
# Expected: ['product', 'great', 'highli', 'recommend']

In [ ]:
# Demo: Apply clean_text() to a DataFrame column
#
# .apply() runs a function on every row of a column, one at a time.
# It's like a for loop, but cleaner and faster.
#
# Before: each row has a raw text string
# After:  each row has a list of cleaned, stemmed words

demo_df = pd.DataFrame({
    'review': [
        'The products are GREAT!!!',
        'Broke after one week. Terrible!'
    ]
})

# .apply(clean_text) runs clean_text() on every row
demo_df['tokens'] = demo_df['review'].apply(clean_text)

print("Before and after:")
print(demo_df[['review', 'tokens']].to_string(index=False))

In [ ]:
# Demo: Count word frequencies
#
# After cleaning, each row has a list of words like ['product', 'great'].
# To count which words appear most across ALL reviews, we need to:
#   1. Flatten all the lists into one big list (using .explode())
#   2. Count how often each word appears (using .value_counts())
#
# .explode() takes a column of lists and turns each list item into its own row:
#   Before: Row 0 -> ['great', 'quality']    Row 1 -> ['love', 'quality']
#   After:  Row 0 -> 'great'   Row 1 -> 'quality'   Row 2 -> 'love'   Row 3 -> 'quality'

demo_reviews = pd.DataFrame({
    'review': [
        'Great quality and great price!',
        'Love the quality. Amazing product.',
        'Good price for the quality.'
    ]
})
demo_reviews['tokens'] = demo_reviews['review'].apply(clean_text)

# Flatten all token lists into individual words
all_words = demo_reviews['tokens'].explode()

print("All words (flattened):")
print(all_words.tolist())
print()
print("Word frequencies (most common first):")
print(all_words.value_counts())

### 📝 Activity 2a: Write a `clean_text()` Function

Write your own `clean_text()` function that takes a string and returns a list of cleaned tokens.

**Your function should do these 5 steps (in order):**
1. Convert to lowercase with `.lower()`
2. Remove all punctuation with `re.sub(r'[^\w\s]', '', text)`
3. Split into words with `word_tokenize(text)`
4. Remove stopwords with a list comprehension: `[w for w in tokens if w not in stop_words]`
5. Stem each word with: `[stemmer.stem(w) for w in tokens]`

*Hint: The demo cell three cells above has the exact code — type it out yourself to practice.*

In [ ]:
# TODO: Write the clean_text() function
# def clean_text(text):
#     text = text.lower()
#     text = re.sub(r'[^\w\s]', '', text)
#     tokens = word_tokenize(text)
#     tokens = [w for w in tokens if w not in stop_words]
#     tokens = [stemmer.stem(w) for w in tokens]
#     return tokens

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [stemmer.stem(w) for w in tokens]
    return tokens

# Test your function with these examples:
# test1 = clean_text("The products are GREAT!!! Highly recommended.")
# print("Test 1:", test1)
# Expected: ['product', 'great', 'highli', 'recommend']
test1 = clean_text("The products are GREAT!!! Highly recommended.")
print("Test 1:", test1)

# test2 = clean_text("Broke after one week. Terrible experience!!!")
# print("Test 2:", test2)
# Expected: ['broke', 'one', 'week', 'terribl', 'experi']
test2 = clean_text("Broke after one week. Terrible experience!!!")
print("Test 2:", test2)

### 📝 Activity 2b: Find the Top 20 Words in Reviews

Apply your `clean_text()` function to the reviews below and find the 20 most common words.

**Steps:**
1. Use `.apply(clean_text)` to clean every review
2. Use `.explode()` to flatten all the word lists into one big column
3. Use `.value_counts().head(20)` to get the top 20

In [ ]:
reviews = pd.DataFrame({
    'review': [
        'Great product! The quality is amazing and the price is very reasonable.',
        'Terrible experience. The product broke after just one week of use.',
        'I love this product! Best purchase I have made in years. Highly recommend!',
        'Not worth the money. Poor quality and terrible customer service.',
        'Amazing quality for the price. Fast shipping and great packaging.',
        'The product is okay but nothing special. Average quality at best.',
        'Excellent product! Great value for money. Would buy again.',
        'Disappointed with the quality. Expected much better for this price.',
        'Fast delivery and the product works perfectly. Very happy with purchase.',
        'Worst product I have ever bought. Complete waste of money.',
        'Really good quality and excellent customer service. Five stars!',
        'The product arrived damaged but customer service replaced it quickly.',
        'Great product for the price! Easy to use and looks great.',
        'Average product. Does the job but could be better quality.',
        'Absolutely love this! Perfect quality and amazing customer support.'
    ]
})

print(f"Number of reviews: {len(reviews)}")
print()

# TODO: Apply clean_text to every review
# reviews['tokens'] = reviews['review'].apply(clean_text)

# TODO: Flatten all token lists and count the top 20
# all_words = reviews['tokens'].explode()
# top_20 = all_words.value_counts().head(20)

reviews['tokens'] = reviews['review'].apply(clean_text)
all_words = reviews['tokens'].explode()
top_20 = all_words.value_counts().head(20)

# Expected: 'product', 'quality', 'great' should be near the top
print("Top 20 words:")
print(top_20)
# print("Top 20 words:")
# print(top_20)

### 📝 Activity 2c: Compare Before and After Stopword Removal

This activity shows **why stopword removal matters**. You'll create two word counts:
- **Without** stopword removal (just lowercase + remove punctuation + tokenize)
- **With** stopword removal (the full `clean_text()` function)

**Steps:**
1. Write a `tokenize_only()` function that does steps 1–3 but skips step 4 (no stopword removal)
2. Get the top 10 words for each version and compare

In [ ]:
# TODO: Write a function that does NOT remove stopwords
def tokenize_only(text):
    """Lowercase, remove punctuation, tokenize -- but keep all words."""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return word_tokenize(text)

# TODO: Get top 10 words for each version
# words_raw = reviews['review'].apply(tokenize_only).explode()
# words_clean = reviews['review'].apply(clean_text).explode()
words_raw = reviews['review'].apply(tokenize_only).explode()
words_clean = reviews['review'].apply(clean_text).explode()
# top_raw = words_raw.value_counts().head(10)
# top_clean = words_clean.value_counts().head(10)
top_raw = words_raw.value_counts().head(10)
top_clean = words_clean.value_counts().head(10)

# Expected: WITHOUT stopwords, the top words will be 'the', 'and', 'is' -- boring!
#           WITH stopwords removed, the top words will be 'product', 'quality', 'great' -- useful!

print("WITHOUT stopword removal (top 10):")
print(top_raw)
print()
print("WITH stopword removal (top 10):")
print(top_clean)

---
# 🏠 AFTER-CLASS ACTIVITIES

Complete these on your own after lecture (~30 min total).

---

## Activity 3: Regex Challenges (~10 min)

Practice your regex skills on trickier patterns. These exercises push beyond the basics.

### Demo: Extracting Phone Numbers and Hashtags

Phone numbers can appear in many formats (555-123-4567, (555) 123-4567, 555.123.4567). We need a pattern flexible enough to handle all of them.

**Pattern breakdown:** `r'\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}'`
- `\(?` — an optional `(` (the `?` makes it optional)
- `\d{3}` — exactly 3 digits (area code)
- `\)?` — an optional `)`
- `[-.\s]?` — an optional separator: dash, dot, or space
- `\d{3}` — 3 more digits
- `[-.\s]?` — another optional separator
- `\d{4}` — the final 4 digits

In [ ]:
# Demo: Phone number extraction
text = "Call us at 555-123-4567 or (555) 987-6543 or 555.111.2222"

phones = re.findall(r'\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}', text)
print("Phone numbers found:", phones)
print()

# Demo: Hashtag extraction
# #\w+ means: a # symbol followed by one or more word characters
tweet = "Loving this new #DataScience course! #Python #NLP #TextMining"
hashtags = re.findall(r'#\w+', tweet)
print("Hashtags found:", hashtags)

### 📝 Activity 3a: Extract Phone Numbers

Extract all phone numbers from the text below. They appear in different formats.

*Hint: Copy the pattern from the demo above: `r'\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}'`*

In [ ]:
contacts = """Customer Service: 800-555-1234
Sales Department: (617) 555-9876
Technical Support: 401.555.3333
Emergency Line: 888-555-0000
Fax (do not call): (212) 555-4444"""

# TODO: Extract all phone numbers
# phones = re.findall(r'\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}', contacts)

phones = re.findall(r'\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}', contacts)
# Expected: 5 phone numbers found

print(f"Found {len(phones)} phone numbers:")
for p in phones:
    print(f"  {p}")

### 📝 Activity 3b: Extract Hashtags from Social Media Posts

Extract all hashtags (words starting with `#`) from these posts, then find the 5 most common ones.

**Steps:**
1. Use `.str.findall(r'#\w+')` to extract hashtags from each post
2. Use `.explode()` to flatten all hashtags into one column
3. Use `.value_counts().head(5)` to get the top 5

In [ ]:
posts = pd.DataFrame({
    'text': [
        'Just finished a great #DataScience project! #Python #Analytics',
        'Learning #NLP is so cool! #DataScience #TextMining',
        'New blog post about #Python and #DataScience for beginners',
        '#Analytics is the future! Love working with #Python',
        'Excited about our #TextMining results! #NLP #DataScience',
        'Best #Python library for #NLP? Definitely NLTK! #DataScience'
    ]
})

# TODO: Extract all hashtags from each post
posts['hashtags'] = posts['text'].str.findall(r'#\w+')

# TODO: Flatten all hashtags and find top 5
all_hashtags = posts['hashtags'].explode()
top_5 = all_hashtags.value_counts().head(5)


# Expected top 5:
# #DataScience (5), #Python (4), #NLP (3), #TextMining (2), #Analytics (2)

print("Top 5 hashtags:")
print(top_5)

---
## Activity 4: Web Scraping + Text Pipeline (~20 min)

So far we've worked with text that was already in a DataFrame. But in the real world, text often lives on **web pages** — news articles, Wikipedia, product pages, etc.

**Web scraping** means using Python to automatically download a web page and extract the text from it. In this activity you'll scrape a Wikipedia page and run your `clean_text()` pipeline on it.

### Background: What is HTML?

Every web page you see in your browser is actually written in a language called **HTML** (HyperText Markup Language). HTML uses **tags** to organize content:

```html
<h1>This is a heading</h1>
<p>This is a paragraph of text.</p>
<a href="https://example.com">This is a link</a>
```

When you visit a web page, your browser reads the HTML and renders it as the nice-looking page you see. When we scrape, we download that raw HTML and use Python to pull out just the text we want.

**You don't need to learn HTML** — just know that `<p>` tags contain paragraphs and `<h2>` tags contain section headings. Python does the hard work for us.

### Demo: Web Scraping with `requests` and `BeautifulSoup`

We need two libraries:
- **`requests`** — downloads a web page (like clicking a link, but in Python)
- **`BeautifulSoup`** — reads the downloaded HTML and helps you find specific parts (paragraphs, headings, links, etc.)

Think of it like this:
- `requests` = "go get the page"
- `BeautifulSoup` = "now find the paragraphs in it"

In [ ]:
# Install the libraries we need for web scraping
# (requests is usually pre-installed, but beautifulsoup4 might not be)
!pip install requests beautifulsoup4

In [ ]:
import requests
from bs4 import BeautifulSoup  # bs4 is the package name, BeautifulSoup is the tool inside it

In [ ]:
# Step 1: Download a web page
#
# requests.get(url) sends a request to the website and downloads the page.
# It returns a "response" object. The status code tells us if it worked:
#   200 = success (the page was downloaded)
#   404 = page not found
#   403 = access denied
#
# When your browser visits a website, it introduces itself (e.g., "I'm Chrome").
# Python doesn't do this by default, so some websites block the request (403).
# We add a User-Agent header to politely tell the website who we are.

url = "https://en.wikipedia.org/wiki/Natural_language_processing"
headers = {"User-Agent": "BryantUniversity-AA640/1.0"}  # identify ourselves to the website
response = requests.get(url, headers=headers)

print(f"Status code: {response.status_code}")  # should be 200
print(f"Page size: {len(response.text):,} characters of HTML")
print()
print("First 300 characters of raw HTML (this is what the browser sees):")
print(response.text[:300])
print("...")

In [ ]:
# Step 2: Parse the HTML with BeautifulSoup
#
# BeautifulSoup reads the messy HTML and organizes it so we can
# easily find specific parts. We tell it to use 'html.parser'
# (Python's built-in HTML reader).

soup = BeautifulSoup(response.text, 'html.parser')

# Now we can easily find things:
print("Page title:", soup.title.text)
print()

# Find all section headings (h2 tags)
headings = soup.find_all('h2')
print("Section headings on this page:")
for h in headings[:8]:
    print(f"  - {h.text.strip()}")

In [ ]:
# Step 3: Extract just the paragraph text
#
#soup.find_all('p') finds every paragraph on the page.
# Each paragraph is wrapped in <p>...</p> tags in HTML.
#
# .text extracts just the human-readable words, removing all the
# HTML tags, links, and formatting code.

paragraphs = soup.find_all('p')
print(f"Found {len(paragraphs)} paragraphs on this page")
print()

# Join all paragraph text into one big string
all_text = ' '.join([p.text for p in paragraphs])
print(f"Total text extracted: {len(all_text):,} characters")
print()
print("First 500 characters of extracted text:")
print(all_text[:500])

In [ ]:
# Step 4: Run our clean_text() pipeline on the scraped text!
#
# This is the payoff: we take raw text from the web and turn it into
# a clean list of meaningful words that we can analyze.

tokens = clean_text(all_text)

print(f"Total tokens after cleaning: {len(tokens)}")
print()

# Count word frequencies
word_counts = pd.Series(tokens).value_counts()
print("Top 15 words on the NLP Wikipedia page:")
print(word_counts.head(15))

### 📝 Activity 4a: Scrape and Analyze a Wikipedia Page

Pick a different Wikipedia page (or use the one provided below). Follow the same 4 steps from the demo:

1. **Download** the page with `requests.get(url)`
2. **Parse** with `BeautifulSoup(response.text, 'html.parser')`
3. **Extract** paragraphs with `soup.find_all('p')` and join them
4. **Clean and count** with `clean_text()` and `.value_counts()`

In [ ]:
# TODO: Scrape a Wikipedia page and find the top 20 words
# url = "https://en.wikipedia.org/wiki/Data_science"

# Step 1: Download the page
# We need a User-Agent header so the website knows who is making the request.
# Without it, Wikipedia will block us with a 403 error.
headers = {"User-Agent": "BryantUniversity-AA640/1.0"}
response = requests.get(url, headers=headers)
print(f"Status: {response.status_code}")

# Step 2: Parse the HTML
soup = BeautifulSoup(response.text, 'html.parser')
print(f"Page: {soup.title.text}")

# Step 3: Extract paragraph text
paragraphs = soup.find_all('p')
all_text = ' '.join([p.text for p in paragraphs])
print(f"Extracted {len(all_text):,} characters")

# Step 4: Clean and count
tokens = clean_text(all_text)
top_20 = pd.Series(tokens).value_counts().head(20)
print()
print("Top 20 words:")
print(top_20)


---
## Summary

Here's what you practiced in this notebook:

| Activity | What you did | Key tools |
|----------|-------------|--------|
| 1a | Extracted years from text | `.str.findall(r'\d{4}')` |
| 1b | Removed punctuation | `.str.replace(r'[^\w\s]', '', regex=True)` |
| 1c | Extracted email addresses | `re.findall(r'\w+@\w+\.\w+')` |
| 2a | Built a `clean_text()` function | `re.sub()`, `word_tokenize()`, stopwords, stemming |
| 2b | Found top 20 words in reviews | `.apply()`, `.explode()`, `.value_counts()` |
| 2c | Compared with/without stopwords | Saw why stopword removal matters |
| 3a | Extracted phone numbers | `re.findall()` with a flexible pattern |
| 3b | Extracted and counted hashtags | `r'#\w+'`, `.value_counts()` |
| 4a | Scraped and analyzed a web page | `requests`, `BeautifulSoup`, `clean_text()` |

**Key takeaway:** The text pipeline (clean → tokenize → remove stopwords → stem) is your starting point for Project #2!

In L9 we'll use this pipeline to count words, create word clouds, and build TF-IDF representations.

---
## Reminders

- **Submit** this notebook via Canvas by the deadline.
- **Office hours**: Check the syllabus for times and location.
- **Project #1**: Presentations are on April 6. Make sure your team is prepared!